# Profile the raw booking data

This notebook profiles the raw hotel bookings before cleaning or modeling. It does not evaluate client claims or choose model features; those are later phases.

No rows are removed or changed here.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import data


In [2]:
bookings = data.read_raw()
print(f"Rows: {len(bookings):,}")
print(f"Columns: {bookings.shape[1]}")
print(f"Raw data path: {data.download()}")


Rows: 119,390
Columns: 32
Raw data path: D:\Aayush\IIT Gandhinagar\C2_W06_Build2_Student_Pack\C2_W06_Build2_Student_Pack\Project_Oversell_IITGN\data\hotels.csv


In [3]:
display(bookings.head())
display(bookings.tail())
print("Column names:")
print(list(bookings.columns))

type_profile = pd.DataFrame({
    "dtype": bookings.dtypes.astype(str),
    "non_null": bookings.notna().sum(),
    "missing": bookings.isna().sum(),
    "missing_pct": (bookings.isna().mean() * 100).round(2),
    "unique_values": bookings.nunique(dropna=False),
})
display(type_profile)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07
119389,City Hotel,0,205,2017,August,35,29,2,7,2,...,No Deposit,9.0,NaN,0,Transient,151.20,0,2,Check-Out,2017-09-07


Column names:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


,dtype,non_null,missing,missing_pct,unique_values
hotel,str,119390,0,0.00,2
is_canceled,int64,119390,0,0.00,2
lead_time,int64,119390,0,0.00,479
arrival_date_year,int64,119390,0,0.00,3
arrival_date_month,str,119390,0,0.00,12
arrival_date_week_number,int64,119390,0,0.00,53
arrival_date_day_of_month,int64,119390,0,0.00,31
stays_in_weekend_nights,int64,119390,0,0.00,17
stays_in_week_nights,int64,119390,0,0.00,35
adults,int64,119390,0,0.00,14


In [4]:
missing_profile = (
    bookings.isna().sum().rename("missing_rows").to_frame()
    .assign(missing_pct=lambda frame: (frame["missing_rows"] / len(bookings) * 100).round(2))
    .query("missing_rows > 0")
    .sort_values("missing_rows", ascending=False)
)
print(f"Columns with missing values: {len(missing_profile)}")
display(missing_profile)

cardinality = bookings.nunique(dropna=False).rename(
    "unique_values_including_missing"
).to_frame().sort_values("unique_values_including_missing")
display(cardinality)


Columns with missing values: 4


,missing_rows,missing_pct
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


,unique_values_including_missing
hotel,2
is_canceled,2
is_repeated_guest,2
arrival_date_year,3
deposit_type,3
reservation_status,3
customer_type,4
babies,5
meal,5
distribution_channel,5


In [5]:
categorical_columns = bookings.select_dtypes(include=["object", "category"]).columns
for column in categorical_columns:
    print(f"\n{column} ({bookings[column].nunique(dropna=False)} values)")
    display(bookings[column].value_counts(dropna=False).head(20).rename("rows").to_frame())

print("Cancellation values:")
display(bookings["is_canceled"].value_counts(dropna=False).rename("rows").to_frame())



hotel (2 values)


C:\Users\aayus\AppData\Local\Temp\ipykernel_22300\448323085.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = bookings.select_dtypes(include=["object", "category"]).columns


,rows
hotel,
City Hotel,79330
Resort Hotel,40060



arrival_date_month (12 values)


,rows
arrival_date_month,
August,13877
July,12661
May,11791
October,11160
April,11089
June,10939
September,10508
March,9794
February,8068



meal (5 values)


... (17 outputs left out)


reservation_status_date (926 values)


,rows
reservation_status_date,
2015-10-21,1461
2015-07-06,805
2016-11-25,790
2015-01-01,763
2016-01-18,625
2015-07-02,469
2016-12-07,450
2015-12-18,423
2016-02-09,412


Cancellation values:


,rows
is_canceled,
0,75166
1,44224


In [6]:
bookings_with_dates = bookings.assign(arrival_date=data.arrival_date(bookings))
print(f"Arrival dates: {bookings_with_dates['arrival_date'].min().date()} "
      f"to {bookings_with_dates['arrival_date'].max().date()}")

print(f"Exact duplicate rows: {bookings.duplicated().sum():,}")
print(f"Rows in duplicate groups: {bookings.duplicated(keep=False).sum():,}")


Arrival dates: 2015-07-01 to 2017-08-31
Exact duplicate rows: 31,994
Rows in duplicate groups: 40,165


In [7]:
nights = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
profile_checks = pd.Series({
    "zero_night_bookings": int((nights == 0).sum()),
    "negative_night_bookings": int((nights < 0).sum()),
    "zero_adr_bookings": int((bookings["adr"] == 0).sum()),
    "negative_adr_bookings": int((bookings["adr"] < 0).sum()),
    "negative_lead_time": int((bookings["lead_time"] < 0).sum()),
    "negative_adults": int((bookings["adults"] < 0).sum()),
    "negative_children_after_blank_as_zero": int((bookings["children"].fillna(0) < 0).sum()),
    "negative_babies": int((bookings["babies"] < 0).sum()),
})
display(profile_checks.rename("rows").to_frame())
display(bookings.select_dtypes(include="number").describe().T)


,rows
zero_night_bookings,715
negative_night_bookings,0
zero_adr_bookings,1959
negative_adr_bookings,1
negative_lead_time,0
negative_adults,0
negative_children_after_blank_as_zero,0
negative_babies,0


,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


## Phase 3 handoff

Use these outputs as observations for `docs/DECISIONS.md`. Do not make cleaning decisions in this notebook. In the next phase, each cleaning or filtering decision must be justified with the observed count and logged before implementation.